In [2]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
from rdkit.Chem import rdMolDescriptors

# Comparar Tanimoto entre clean_smiles y noised_smiles en (1), (0.5) y (0.25)

def read_smiles_list(path):
    with open(path, 'r') as f:
        return [line.strip() for line in f if line.strip()]

def mol_to_fp(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=2048)

def pairwise_tanimoto(clean_smiles, noised_smiles):
    n = min(len(clean_smiles), len(noised_smiles))
    scores = []
    for i in range(n):
        #check that clean smiles and noised smiles have the same molecular formula.
        mol1 = Chem.MolFromSmiles(clean_smiles[i])
        mol2 = Chem.MolFromSmiles(noised_smiles[i])
        if mol1 is None or mol2 is None:
            continue
        if rdMolDescriptors.CalcMolFormula(mol1) != rdMolDescriptors.CalcMolFormula(mol2):
            print(f"Molecular formula mismatch for {clean_smiles[i], rdMolDescriptors.CalcMolFormula(mol1)} and {noised_smiles[i], rdMolDescriptors.CalcMolFormula(mol2) }")
            #continue
        fp1 = mol_to_fp(clean_smiles[i])
        fp2 = mol_to_fp(noised_smiles[i])
        if fp1 is None or fp2 is None:
            continue
        scores.append(DataStructs.TanimotoSimilarity(fp1, fp2))
    return np.array(scores, dtype=float)

df = pd.read_csv('../mols_gen/251104_database_allmolecules_main_2_22_confps_timepred_2_22_confps_seed/all_generated_molecules.csv')
clean_mols = df.original_smiles.to_list()
noise_mols = df.smiles.to_list()
results = []

lvl = "generated"
scores = pairwise_tanimoto(clean_mols, noise_mols)

print(
    f"Nivel {lvl}: n={scores.size} | media={scores.mean():.4f} | mediana={np.median(scores):.4f} | p10={np.percentile(scores,10):.4f} | p90={np.percentile(scores,90):.4f}"
)
results.append(pd.DataFrame({'noise_level': lvl, 'tanimoto': scores}))

if results:
    df_tanimoto = pd.concat(results, ignore_index=True)
    display(df_tanimoto.groupby('noise_level')['tanimoto'].describe())
    out_csv = '../Data/tanimoto_clean_vs_noised.csv'
    df_tanimoto.to_csv(out_csv, index=False)
    print(f"Guardado CSV con resultados en {out_csv}")
else:
    print("No se generaron resultados de Tanimoto.")


Molecular formula mismatch for ('C=C(C)C(=O)OC1CC(C)(C)N([O])C(C)(C)C1', 'C13H22NO3') and ('O=C(O)CCCCCCC(=O)N1CCCCC1', 'C13H23NO3')
Molecular formula mismatch for ('CN(C)c1ccc(C2=[N+]([O-])C(C)(C)C(C)(C)N2[O])cc1', 'C15H22N3O2') and ('CC(C)(C)CN1CCN(c2ccc3c(c2)[N+]3([O-])O)CC1', 'C15H23N3O2')
Molecular formula mismatch for ('CCCCCCCC[Al]CCCCCCCC', 'C16H34Al') and ('CCCCCCCCCCCCCC[AlH]CC', 'C16H35Al')
Molecular formula mismatch for ('COc1ccc(C2=NC(C)(C)C(C)(C)N2[O])cc1', 'C14H19N2O2') and ('O=C(O)N1CCN(CCCc2ccccc2)CC1', 'C14H20N2O2')
Molecular formula mismatch for ('[CH2]CCCC', 'C5H11') and ('CCCCC', 'C5H12')
Molecular formula mismatch for ('CCCCCCCCCCCCC[C]=O', 'C14H27O') and ('CC(C)(C)OC1CCCCCCCCC1', 'C14H28O')
Molecular formula mismatch for ('N[Rh+3](N)(N)(N)(N)N', 'H12N6Rh+3') and ('NN(N)[Rh+3](N)(N)N', 'H10N6Rh+3')
Molecular formula mismatch for ('CCCCN(CCCC)C1CC(C)(C)N([O])C(C)(C)C1', 'C17H35N2O') and ('CC(C)(C)N(O)CCCCCCCCN1CCCCC1', 'C17H36N2O')
Molecular formula mismatch for ('

,count,mean,std,min,25%,50%,75%,max
noise_level,,,,,,,,
generated,80000.0,0.131997,0.068963,0.0,0.09901,0.122302,0.150442,1.0


Guardado CSV con resultados en ../Data/tanimoto_clean_vs_noised.csv


In [3]:
import pandas as pd

datos_noised = pd.read_csv("../mols_gen/noised_smiles/noised_smiles.csv")

In [4]:
# Filter rows where step equals total_steps (final denoised molecules)
clean_data = datos_noised[datos_noised['step'] == datos_noised['total_steps']]

# filter rows where sigma_norm is the closer to 0.25 for that mol_index
#list_of_index = datos_noised.groupby('mol_index')['sigma_norm'].transform(lambda x: (x - 0.0075).abs().idxmin())
# get unique values
#unique_values = list_of_index.unique()
# filter rows selecting the indexes on unique_values
#clean_data = datos_noised.iloc[unique_values]

# Get the clean SMILES list
clean_smiles_list = clean_data['smiles'].tolist()

# Get the noised SMILES list (all rows)
noised_smiles_list = clean_data['noised_smiles'].tolist()

# Save clean SMILES to file
with open('../Data/clean_smiles.smiles', 'w') as f:
    for smiles in clean_smiles_list:
        f.write(f"{smiles}\n")

# Save noised SMILES to file
with open('../Data/noised_smiles_(0.5).smiles', 'w') as f:
    for smiles in noised_smiles_list:
        f.write(f"{smiles}\n")

print(f"Saved {len(clean_smiles_list)} clean SMILES to clean_smiles.smiles")
print(f"Saved {len(noised_smiles_list)} noised SMILES to noised_smiles.smiles")


Saved 20000 clean SMILES to clean_smiles.smiles
Saved 20000 noised SMILES to noised_smiles.smiles


In [5]:

from typing import List
import argparse
import os
import numpy as np
from pickle import load


from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger

import pandas as pd

class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]

setup_default_logger()

description='Molecule distribution learning benchmark for random smiles sampler'
dist_file='../Data/clean_smiles.smiles'
output_dir = None
suite='v2'

if output_dir is None:
    output_dir = os.getcwd()


with open('../Data/noised_smiles_(0.5).smiles', 'r') as f:
    smiles_list = f.readlines()



generator = RandomSmilesSampler(molecules=smiles_list)


json_file_path = os.path.join(output_dir, '../Data/distribution_learning_results.json')


#hacer un print del smiles tambien

assess_distribution_learning(generator,
                             chembl_training_file=dist_file,
                             json_output_file=json_file_path,
                             benchmark_version=suite)


INFO : Benchmarking distribution learning, version v2
INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999500
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9995}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.548464
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.01683074945775962, 'MolLogP': 1.0760261130497677, 'MolWt': 0.0015

In [ ]:

from typing import List
import argparse
import os
import numpy as np
from pickle import load


from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger

import pandas as pd

class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]

setup_default_logger()

description='Molecule distribution learning benchmark for random smiles sampler'
dist_file='../Data/noised_smiles_(0.5).smiles'
output_dir = None
suite='v2'

if output_dir is None:
    output_dir = os.getcwd()


gen_smiles = pd.read_csv("../mols_gen/250211_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit/all_generated_molecules.csv")
smiles_list = gen_smiles.smiles.to_list()


generator = RandomSmilesSampler(molecules=smiles_list)


json_file_path = os.path.join(output_dir, '../Data/distribution_learning_results.json')


#hacer un print del smiles tambien

assess_distribution_learning(generator,
                             chembl_training_file=dist_file,
                             json_output_file=json_file_path,
                             benchmark_version=suite)


INFO : Benchmarking distribution learning, version v2
INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 9992}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.999200
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9992}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.534124
INFO :   Sampling time: 0:00:02
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.011656926307748422, 'MolLogP': 1.0474693549794958, 'MolWt': 0.0013

In [15]:
# Filter rows where step equals total_steps (final denoised molecules)
clean_data = datos_noised[datos_noised['step'] == 1]

# Get the noised SMILES list (all rows)
noised_smiles_list = clean_data['noised_smiles'].tolist()

# Save noised SMILES to file
with open('../Data/noised_smiles_1step.smiles', 'w') as f:
    for smiles in noised_smiles_list:
        f.write(f"{smiles}\n")

print(f"Saved {len(noised_smiles_list)} noised SMILES to noised_smiles.smiles")


Saved 19997 noised SMILES to noised_smiles.smiles


In [1]:
from typing import List
import os
import numpy as np
import pandas as pd

from guacamol.distribution_matching_generator import DistributionMatchingGenerator
from guacamol.assess_distribution_learning import assess_distribution_learning
from guacamol.utils.helpers import setup_default_logger


class RandomSmilesSampler(DistributionMatchingGenerator):
    """
    Generator that samples SMILES strings from a predefined list.
    Shuffles the list and returns the first N samples to avoid duplicates and bias.
    """

    def __init__(self, molecules: List[str]) -> None:
        """
        Args:
            molecules: list of molecules from which the samples will be drawn
        """
        self.molecules = list(np.random.permutation(molecules))
        self.molecules = molecules

    def generate(self, number_samples: int) -> List[str]:
       
        return self.molecules[:number_samples]



# Configure logging
setup_default_logger()

# Benchmark configuration
suite = 'v2'
output_dir = os.getcwd()

with open('../Data/noised_smiles_1step.smiles', 'r') as f:
    smiles_list = f.readlines()
# Source molecules for the generator
#smiles_csv = '../mols_gen/251012_database_allmolecules_main_2_22_confps_timepred_2_22_confps_sinexplicit_unseed2/all_generated_molecules.csv'
#smiles_list = pd.read_csv(smiles_csv).smiles.to_list()

# Run the benchmark across files CID-SMILES-filtered-lt70_0 .. _4
for file_index in range(5):
    dist_file = f"../Data/CID-SMILES-filtered-lt70_{file_index}_nuevo3.txt"
    json_file_path = os.path.join(output_dir, f"../Data/distribution_learning_results_{file_index}.json")

    generator = RandomSmilesSampler(molecules=smiles_list)
    print(f"Running benchmark for {dist_file} → {json_file_path}")

    assess_distribution_learning(
        generator,
        chembl_training_file=dist_file,
        json_output_file=json_file_path,
        benchmark_version=suite,
    )



INFO : Benchmarking distribution learning, version v2


Running benchmark for ../Data/CID-SMILES-filtered-lt70_0_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_0.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.996700
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9967}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.872887
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.08653907317559077, 'MolLogP': 0.015467102297692624, 'MolWt': 0.0409912280161116, 'TPSA': 0.02730119952540796, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_1_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_1.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.872412
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09484649300297822, 'MolLogP': 0.01786021095128003, 'MolWt': 0.04553528196451467, 'TPSA': 0.026929713033246906, 'NumHAcc

...

INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998200
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9982}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.872914
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09486749531630845, 'MolLogP': 0.02182221313539931, 'MolWt': 0.04605981104773869, 'TPSA': 0.02895549110728319, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_3_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_3.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.998000
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9980}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.876527
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.0901543183640151, 'MolLogP': 0.01712054914200714, 'MolWt': 0.04365053466934045, 'TPSA': 0.029610354487286272, 'NumHAcce

Running benchmark for ../Data/CID-SMILES-filtered-lt70_4_nuevo3.txt → /export/home/shared/Projects/Manu22-23/MoleculeDifusion/CoCoGraph/compare_guacamol/../Data/distribution_learning_results_4.json


INFO : Number of benchmarks: 5
INFO : Running benchmark 1/5: Validity
INFO : Results for the benchmark "Validity":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_valid': 10000}
INFO : Running benchmark 2/5: Uniqueness
INFO : Results for the benchmark "Uniqueness":
INFO :   Score: 1.000000
INFO :   Sampling time: 0:00:00
INFO :   Metadata: {'number_samples': 10000, 'number_unique': 10000}
INFO : Running benchmark 3/5: Novelty
INFO : Results for the benchmark "Novelty":
INFO :   Score: 0.997800
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'number_novel': 9978}
INFO : Running benchmark 4/5: KL divergence
INFO : Results for the benchmark "KL divergence":
INFO :   Score: 0.869306
INFO :   Sampling time: 0:00:01
INFO :   Metadata: {'number_samples': 10000, 'kl_divs': {'BertzCT': 0.09740973433341045, 'MolLogP': 0.022697412706629227, 'MolWt': 0.04487442902774137, 'TPSA': 0.031869698772130334, 'NumHAc